In [ ]:
# --- Planilla OCR de Document AI -> una fila por renglón ----------------------
import json, re, unicodedata
from bisect import bisect_right
from decimal import Decimal, InvalidOperation
from statistics import median
import pandas as pd

JSON_PATH = "/content/document (1).json"  # subilo desde el panel de archivos de Colab
ROW_TOL = 0.6       # misma fila si los centros difieren menos que ROW_TOL × alto de token
SHOW_HEADERS = True

# Títulos de izquierda a derecha.
# "left": el contenido arranca donde arranca el título (texto).
# "right": el contenido termina donde termina el título (números).
HEADERS = [
    ("nro_int",      "Nro. Int.",           "left"),
    ("nro_afiliado", "Nro. de Afiliado",    "left"),
    ("nro_bono",     "Nro. de Bono",        "left"),
    ("profesional",  "Profesional Efector", "left"),
    ("servicio",     "Servicio",            "left"),
    ("fecha",        "Fecha",               "left"),
    ("paciente",     "Apellido y Nombre",   "left"),
    ("prestacion",   "Prestación",          "left"),
    ("cant",         "Cant.",               "right"),
    ("der",          "Der.",                "right"),
    ("hon",          "Hon.",                "right"),
    ("total",        "Total",               "right"),
]

# El OCR a veces devuelve letras griegas iguales a las latinas ("ΜΑPA").
GREEK = str.maketrans("ΑΒΕΖΗΙΚΜΝΟΡΤΥΧοα", "ABEZHIKMNOPTYXoa")


def key(s):
    """Versión comparable: sin acentos, minúsculas, solo letras y números."""
    s = unicodedata.normalize("NFKD", s.translate(GREEK))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return re.sub(r"[^a-z0-9]", "", s)


def anchor_text(full, layout):
    """Recorta del texto completo el tramo al que apunta el elemento."""
    segs = (layout.get("textAnchor") or {}).get("textSegments") or []
    return "".join(full[int(s.get("startIndex", 0)):int(s.get("endIndex", 0))] for s in segs)


def page_tokens(full, page):
    """Tokens de la página con su texto y su caja normalizada 0..1."""
    out = []
    for t in page.get("tokens", []):
        lay = t.get("layout", {})
        if lay.get("orientation", 0) not in (0, 1, "PAGE_UP", "ORIENTATION_UNSPECIFIED"):
            continue  # texto rotado (logos, sellos), no es la tabla
        v = (lay.get("boundingPoly") or {}).get("normalizedVertices") or []
        if not v:
            continue
        xs = [p.get("x", 0.0) for p in v]
        ys = [p.get("y", 0.0) for p in v]
        raw = anchor_text(full, lay).replace("\n", " ").translate(GREEK)
        if t.get("detectedBreak") and not raw.endswith(" "):
            raw += " "
        out.append({
            "raw": raw, "key": key(raw), "conf": lay.get("confidence", 0.0),
            "x0": min(xs), "x1": max(xs), "y0": min(ys), "y1": max(ys),
            "xc": (min(xs) + max(xs)) / 2, "yc": (min(ys) + max(ys)) / 2,
        })
    return out


def group_rows(toks):
    """Agrupa tokens en filas visuales según su altura en la página."""
    if not toks:
        return []
    h = median(t["y1"] - t["y0"] for t in toks)
    rows = []
    for t in sorted(toks, key=lambda t: t["yc"]):
        if rows and abs(t["yc"] - rows[-1]["yc"]) <= ROW_TOL * h:
            rows[-1]["toks"].append(t)
            rows[-1]["yc"] = median(x["yc"] for x in rows[-1]["toks"])
        else:
            rows.append({"yc": t["yc"], "toks": [t]})
    for r in rows:
        r["toks"].sort(key=lambda t: t["x0"])
    return rows


def match_headers(toks):
    """Busca los títulos, en orden, dentro de una fila. Devuelve {columna: caja del título}."""
    found, i = {}, 0
    for name, label, align in HEADERS:
        target = key(label)
        for j in range(i, len(toks)):
            if not toks[j]["key"]:
                continue
            acc, k = "", j
            while k < len(toks) and len(acc) < len(target) and target.startswith(acc + toks[k]["key"]):
                acc += toks[k]["key"]
                k += 1
            if acc == target:
                span = [t for t in toks[j:k] if t["key"]]
                found[name] = {"align": align,
                               "x0": min(t["x0"] for t in span), "x1": max(t["x1"] for t in span)}
                i = k
                break
    return found


def column_cuts(found):
    """Límites horizontales entre columnas, según cómo se alinea cada una."""
    cols = sorted(found.items(), key=lambda kv: kv[1]["x0"])
    cuts = []
    for (_, a), (_, b) in zip(cols, cols[1:]):
        if a["align"] == "right" and b["align"] == "right":
            cuts.append(a["x1"])                    # entre números: donde termina el título izquierdo
        elif a["align"] == "right":
            cuts.append((a["x1"] + b["x0"]) / 2)    # número y después texto: a mitad de camino
        else:
            cuts.append(b["x0"])                    # después de texto: donde arranca el título siguiente
    return [n for n, _ in cols], cuts


def clean(s):
    """Saca el ruido de los tachados ('|||', '....') y espacios de más."""
    s = re.sub(r"\s+", " ", s.replace("|", " ")).strip()
    return s.strip(" .")


def parse_page(full, page):
    n_page = page.get("pageNumber")
    rows = group_rows(page_tokens(full, page))
    header = max(rows, key=lambda r: len(match_headers(r["toks"])), default=None)
    found = match_headers(header["toks"]) if header else {}
    if len(found) < len(HEADERS) // 2:
        print(f"Página {n_page}: no encontré la fila de títulos")
        return []
    names, cuts = column_cuts(found)
    if SHOW_HEADERS:
        missing = [n for n, _, _ in HEADERS if n not in found]
        print(f"Página {n_page}: {len(found)} títulos", f"(faltan: {missing})" if missing else "")
    top = max(t["y1"] for t in header["toks"])
    out = []
    for r in rows:
        if r["yc"] <= top:
            continue  # encabezado de la página y fila de títulos
        if re.search(r"pagina\d+de\d+", "".join(t["key"] for t in r["toks"])):
            continue  # pie "Página X de N"
        cells = dict.fromkeys(names, "")
        for t in r["toks"]:
            cells[names[bisect_right(cuts, t["xc"])]] += t["raw"]
        cells = {n: clean(v) for n, v in cells.items()}
        if not any(cells.values()):
            continue
        cells.update(
            pagina=n_page,
            y0=round(min(t["y0"] for t in r["toks"]), 4),
            y1=round(max(t["y1"] for t in r["toks"]), 4),
            conf_min=round(min(t["conf"] for t in r["toks"]), 3),
        )
        out.append(cells)
    return out


def ar_decimal(s):
    """'144283,00' o '1.234,56' -> Decimal; vacío o ilegible -> None."""
    s = (s or "").replace("$", "").replace(" ", "")
    try:
        return Decimal(s.replace(".", "").replace(",", ".")) if s else None
    except InvalidOperation:
        return None


doc = json.load(open(JSON_PATH, encoding="utf-8"))
df = pd.DataFrame([row for page in doc["pages"] for row in parse_page(doc["text"], page)])

df[["codigo", "descripcion"]] = df["prestacion"].str.extract(r"^(\S+)\s*(.*)$")
for c in ["der", "hon", "total"]:
    df[c] = df[c].map(ar_decimal)
df["cant"] = pd.to_numeric(df["cant"], errors="coerce").astype("Int64")
df["fecha_dt"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce")
# Control gratis: en esta planilla Der. + Hon. = Total. Si no da, la fila se leyó o se partió mal.
df["total_ok"] = [all(isinstance(v, Decimal) for v in (d, h, t)) and d + h == t
                  for d, h, t in zip(df["der"], df["hon"], df["total"])]

cols = (["pagina"] + [n for n, _, _ in HEADERS]
        + ["codigo", "descripcion", "fecha_dt", "total_ok", "conf_min", "y0", "y1"])
df = df[[c for c in cols if c in df.columns]]
print(f"{len(df)} renglones · {(~df['total_ok']).sum()} con Der.+Hon. ≠ Total")
df

In [ ]:
# --- Document AI online sobre el PDF: latencia, costo y robustez de la extracción --------------
# Usa las funciones de la celda anterior (page_tokens, group_rows, match_headers, parse_page,
# ar_decimal, HEADERS): corré esa celda antes que esta.
%pip install -q pypdf

import base64, hashlib, io, os, random, time
from decimal import Decimal

import google.auth
import pandas as pd
from google.auth.transport.requests import AuthorizedSession
from pypdf import PdfReader, PdfWriter

PDF_PATH = "/content/detalle_(1)_ANONIMIZADO_sin_afiliado.pdf"
# Datos de GCP: Secretos de Colab (🔑) o variables de entorno. No van escritos en el código.
def secreto(nombre, defecto=""):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre, defecto)


ENDPOINT = secreto("DOCAI_ENDPOINT")   # https://us-documentai.googleapis.com/v1/projects/<PROYECTO>/locations/us/processors/<PROCESADOR>:process
RUNS = 3                  # veces que se procesa el mismo PDF (latencia y determinismo); cada una se cobra
PAGES_PER_REQUEST = 15    # tope del modo online del OCR (30 con IMAGELESS = True)
IMAGELESS = False
# Pedir solo lo que usa el parser achica la respuesta. Si el endpoint lo rechaza, poné None.
FIELD_MASK = "text,pages.pageNumber,pages.dimension,pages.tokens"
MAX_RETRIES = 3
LOW_CONF = 0.80           # tokens con menos confianza que esto cuentan como dudosos

# Enterprise Document OCR, USD por 1.000 páginas. Verificalos en
# https://cloud.google.com/document-ai/pricing antes de usar los números.
PRICE_TIER1 = 1.50        # hasta 5M páginas/mes
PRICE_TIER2 = 0.60        # desde 5M páginas/mes
TARGET_PAGES_PER_DAY = 1_000_000

SHOW_HEADERS = False      # silencia el print por página de parse_page
FIELDS = [n for n, _, _ in HEADERS]
KEY_FIELDS = ["nro_int", "fecha_dt", "codigo", "cant", "total"]

# Reusa la SA del notebook: la variable `credentials` si existe; si no, la credencial por defecto.
creds = globals().get("credentials") or google.auth.default()[0]
if getattr(creds, "requires_scopes", False):
    creds = creds.with_scopes(["https://www.googleapis.com/auth/cloud-platform"])
session = AuthorizedSession(creds)


def split_pdf(path, size):
    """Divide el PDF en tramos de hasta `size` páginas. Devuelve (páginas, [(offset, bytes)])."""
    raw = open(path, "rb").read()
    reader = PdfReader(io.BytesIO(raw))
    n = len(reader.pages)
    if n <= size:
        return n, [(0, raw)]  # el original tal cual, sin reescribirlo
    chunks = []
    for start in range(0, n, size):
        writer = PdfWriter()
        for i in range(start, min(start + size, n)):
            writer.add_page(reader.pages[i])
        buf = io.BytesIO()
        writer.write(buf)
        chunks.append((start, buf.getvalue()))
    return n, chunks


def process_chunk(pdf_bytes):
    """Un request online; reintenta 429/5xx con espera creciente. Devuelve (doc, s, bytes, reintentos)."""
    body = {"rawDocument": {"content": base64.b64encode(pdf_bytes).decode(),
                            "mimeType": "application/pdf"}}
    if IMAGELESS:
        body["imagelessMode"] = True
    if FIELD_MASK:
        body["fieldMask"] = FIELD_MASK
    for attempt in range(MAX_RETRIES + 1):
        t0 = time.perf_counter()
        resp = session.post(ENDPOINT, json=body, timeout=300)
        elapsed = time.perf_counter() - t0
        if resp.ok:
            return resp.json()["document"], elapsed, len(resp.content), attempt
        if resp.status_code not in (429, 500, 502, 503, 504) or attempt == MAX_RETRIES:
            raise RuntimeError(f"Document AI {resp.status_code}: {resp.text[:500]}")
        time.sleep(2 ** attempt + random.random())


def is_empty(v):
    return v is None or (isinstance(v, str) and not v.strip()) or (not isinstance(v, str) and pd.isna(v))


def to_table(docs):
    """Renglones de todos los tramos, con la misma lógica que la celda anterior."""
    rows = []
    for offset, doc in docs:
        for page in doc.get("pages", []):
            for r in parse_page(doc.get("text", ""), page):
                r["pagina"] = offset + page.get("pageNumber", 0)
                rows.append(r)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if "prestacion" in df:
        df[["codigo", "descripcion"]] = df["prestacion"].str.extract(r"^(\S+)\s*(.*)$")
    for c in ["der", "hon", "total"]:
        if c in df:
            df[c] = df[c].map(ar_decimal)
    if "cant" in df:
        df["cant"] = pd.to_numeric(df["cant"], errors="coerce").astype("Int64")
    if "fecha" in df:
        df["fecha_dt"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce")
    money = [df[c] if c in df else [None] * len(df) for c in ("der", "hon", "total")]
    df["total_ok"] = [all(isinstance(v, Decimal) for v in trio) and trio[0] + trio[1] == trio[2]
                      for trio in zip(*money)]
    return df


def quality(docs, df):
    """Indicadores de robustez de una corrida."""
    confs, headers = [], []
    for _, doc in docs:
        for page in doc.get("pages", []):
            toks = page_tokens(doc.get("text", ""), page)
            confs += [t["conf"] for t in toks]
            headers.append(max((len(match_headers(r["toks"])) for r in group_rows(toks)), default=0))
    q = {
        "renglones": len(df),
        "total_ok_%": round(100 * df["total_ok"].mean(), 1) if "total_ok" in df else 0.0,
        "paginas_con_todos_los_titulos": f"{sum(h == len(HEADERS) for h in headers)}/{len(headers)}",
        "tokens_dudosos_%": round(100 * sum(c < LOW_CONF for c in confs) / max(len(confs), 1), 2),
        "conf_min": round(min(confs), 3) if confs else None,
    }
    for f in KEY_FIELDS:
        q[f"sin_{f}"] = int(df[f].map(is_empty).sum()) if f in df else len(df)
    return q


def row_set(df):
    cols = [c for c in FIELDS if c in df]
    return set(map(tuple, df[cols].astype(str).values)) if len(df) else set()


def signature(df):
    cols = [c for c in FIELDS if c in df]
    if not len(df):
        return "vacía"
    return hashlib.sha256(df[cols].astype(str).to_csv(index=False).encode()).hexdigest()[:12]


n_pages, chunks = split_pdf(PDF_PATH, PAGES_PER_REQUEST)
print(f"{n_pages} páginas en {len(chunks)} request(s) por corrida, {RUNS} corridas\n")

runs, tables = [], []
for run in range(1, RUNS + 1):
    t_run = time.perf_counter()
    docs, ocr_s, size, retries = [], 0.0, 0, 0
    for offset, data in chunks:
        doc, secs, nbytes, n_retry = process_chunk(data)
        docs.append((offset, doc))
        ocr_s += secs
        size += nbytes
        retries += n_retry
    t_parse = time.perf_counter()
    df_run = to_table(docs)
    parse_s = time.perf_counter() - t_parse
    total_s = time.perf_counter() - t_run
    tables.append(df_run)
    runs.append({
        "corrida": run, "total_s": round(total_s, 2), "ocr_s": round(ocr_s, 2),
        "parseo_s": round(parse_s, 3), "s_por_pagina": round(total_s / n_pages, 3),
        "respuesta_MB": round(size / 1e6, 2), "reintentos": retries, "tabla": signature(df_run),
        **quality(docs, df_run),
    })

report = pd.DataFrame(runs).set_index("corrida")
display(report)

# Latencia. La corrida 1 suele ser más lenta: incluye abrir la conexión y pedir el token.
lat = report["total_s"]
print(f"\nLatencia por PDF: mín {lat.min():.2f}s · mediana {lat.median():.2f}s · máx {lat.max():.2f}s "
      f"→ {lat.median() / n_pages:.2f}s por página (parseo local: {report['parseo_s'].median():.3f}s)")

# Costo: Document AI cobra por página procesada.
per_pdf = n_pages * PRICE_TIER1 / 1000
print(f"Costo: USD {per_pdf:.4f} por PDF · USD {per_pdf * RUNS:.4f} esta prueba ({RUNS} corridas)")
month = TARGET_PAGES_PER_DAY * 30
monthly = (min(month, 5_000_000) * PRICE_TIER1 + max(month - 5_000_000, 0) * PRICE_TIER2) / 1000
print(f"A {TARGET_PAGES_PER_DAY:,} páginas/día: ~USD {monthly:,.0f}/mes (~USD {monthly / 30:,.0f}/día) solo de OCR")

# Paralelismo necesario para ese volumen, con requests del mismo tamaño que los de esta prueba.
pages_per_req = n_pages / len(chunks)
req_s = report["ocr_s"].median() / len(chunks)
target_pps = TARGET_PAGES_PER_DAY / 86_400
print(f"Para ~{target_pps:.1f} págs/s hacen falta ~{target_pps / (pages_per_req / req_s):.0f} requests "
      f"en paralelo de {pages_per_req:.0f} páginas ({req_s:.2f}s cada uno)")

# Determinismo: ¿el mismo PDF da la misma tabla en cada corrida?
if report["tabla"].nunique() == 1:
    print("Determinismo: todas las corridas dieron exactamente la misma tabla")
else:
    base = row_set(tables[0])
    for i, t in enumerate(tables[1:], start=2):
        print(f"Determinismo: la corrida {i} tiene {len(row_set(t) ^ base)} renglones distintos de la corrida 1")

cols = ["pagina"] + FIELDS + ["codigo", "descripcion", "fecha_dt", "total_ok", "conf_min", "y0", "y1"]
df_ocr = tables[-1][[c for c in cols if c in tables[-1].columns]]
if "total_ok" in df_ocr and (~df_ocr["total_ok"]).any():
    print("\nRenglones donde Der. + Hon. ≠ Total:")
    display(df_ocr[~df_ocr["total_ok"]])
df_ocr

In [ ]:
# --- Juez de idempotencia con Gemini 3.8 Flash sobre las corridas de la celda anterior -----------
# Usa `tables`, `FIELDS` y `creds` de las celdas anteriores.
# El código compara las corridas celda por celda (exacto y gratis); el juez solo decide si cada
# diferencia cambia un dato que importa.
%pip install -q google-genai

import json, time
from collections import Counter
from decimal import Decimal
from typing import Literal

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel

MODEL = "gemini-3.8-flash"
PROJECT_ID = getattr(creds, "project_id", None) or secreto("GCP_PROJECT")
LOCATION = "global"
MAX_DIFFS = 200                 # diferencias que se le mandan al juez como máximo
CALL_JUDGE_IF_IDENTICAL = True  # en producción, False: sin diferencias no hace falta pagar el juez
INJECT_TEST_DIFFS = False       # True: agrega 3 diferencias de prueba a la corrida 2 para ver al juez
# USD por 1M tokens de gemini-3.8-flash (tabla del repo: 0,75/3,75 hasta fin de 2026; 1,50/7,50 desde 2027)
PRICE_IN, PRICE_OUT = 0.75, 3.75
NUMERIC = {"cant", "der", "hon", "total"}


def as_text(v):
    """Valor comparable: importes con 2 decimales, vacíos como ''."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return ""
    return f"{v:.2f}" if isinstance(v, Decimal) else str(v)


def keyed_rows(df):
    """Renglones por (página, Nro. Int.); sin número, por su orden en la página."""
    out, rank, seen = {}, Counter(), Counter()
    for _, r in df.sort_values(["pagina", "y0"]).iterrows():
        rank[r["pagina"]] += 1
        nro = as_text(r.get("nro_int")).strip()
        base = f"pág. {r['pagina']} · " + (f"Nro. Int. {nro}" if nro else f"renglón {rank[r['pagina']]}")
        seen[base] += 1
        key = base if seen[base] == 1 else f"{base} ({seen[base]})"
        out[key] = {f: as_text(r.get(f)) for f in FIELDS if f in df}
    return out


# Opcional: diferencias conocidas para comprobar que el juez clasifica bien.
judged = list(tables)
if INJECT_TEST_DIFFS and len(judged) > 1 and len(judged[1]) >= 3:
    t = judged[1].copy()
    i0, i1, i2 = t.index[:3]
    t.at[i0, "prestacion"] = str(t.at[i0, "prestacion"]).replace("A", "Α", 1)  # A griega: cosmética
    t.at[i1, "servicio"] = f"{t.at[i1, 'servicio']} ."                           # ruido: cosmética
    if isinstance(t.at[i2, "total"], Decimal):
        t.at[i2, "total"] += 1                                                     # importe: material
    judged[1] = t
    print("Prueba: esperado NO_IDEMPOTENTE (2 diferencias cosméticas y 1 material)\n")

# 1) Diferencias exactas, celda por celda.
rows_by_run = [keyed_rows(t) for t in judged]
diffs = []
for key in sorted(set().union(*rows_by_run)):
    if not all(key in rr for rr in rows_by_run):
        diffs.append({"renglon": key, "campo": "(renglón)",
                      "valores_por_corrida": ["presente" if key in rr else "falta" for rr in rows_by_run]})
        continue
    for f in FIELDS:
        vals = [rr[key].get(f, "") for rr in rows_by_run]
        if len(set(vals)) > 1:
            diffs.append({"renglon": key, "campo": f, "valores_por_corrida": vals})
for i, d in enumerate(diffs, 1):
    d["id"] = i

print(f"Renglones por corrida: {[len(t) for t in judged]} · diferencias exactas: {len(diffs)}")
if diffs:
    print("Por campo:", dict(Counter(d["campo"] for d in diffs)))


# 2) Juez: clasifica cada diferencia y da el veredicto.
class Diferencia(BaseModel):
    id: int
    materialidad: Literal["MATERIAL", "COSMETICA"]
    motivo: str


class Veredicto(BaseModel):
    veredicto: Literal["IDEMPOTENTE", "IDEMPOTENTE_CON_DIFERENCIAS_COSMETICAS", "NO_IDEMPOTENTE"]
    diferencias: list[Diferencia]
    resumen: str


INSTRUCCIONES = """Sos un juez de calidad de un pipeline que extrae renglones de planillas de
facturación médica (OCR + parser). El mismo PDF se procesó varias veces. El pipeline es idempotente
si todas las corridas producen la misma información.

Recibís la comparación exacta que ya hizo el código: cuántos renglones dio cada corrida y cada celda
que difiere, con su valor en cada corrida, en orden. No busques diferencias nuevas: juzgá todas las
listadas, cada una por su id.

Clasificá cada diferencia:
- MATERIAL: cambia un dato que se usa para detectar prácticas duplicadas o calcular débitos
  (Nro. Int., fecha, código de prestación, qué prestación se lee, cantidad, importes), o un renglón
  falta en alguna corrida (campo "(renglón)"). Los importes ya vienen con 2 decimales: si difieren,
  el número es distinto.
- COSMETICA: una persona leería el mismo dato (espacios, mayúsculas, puntuación o un carácter de
  ruido, letras griegas iguales a las latinas).

Veredicto:
- IDEMPOTENTE: no hay diferencias.
- IDEMPOTENTE_CON_DIFERENCIAS_COSMETICAS: hay diferencias y todas son cosméticas.
- NO_IDEMPOTENTE: hay al menos una diferencia material.

Escribí cada motivo y el resumen en español, en una o dos oraciones."""

if not diffs and not CALL_JUDGE_IF_IDENTICAL:
    print("\nVeredicto: IDEMPOTENTE (las corridas son idénticas; no hizo falta el juez)")
else:
    sent = diffs[:MAX_DIFFS]
    payload = {
        "corridas": len(judged),
        "renglones_por_corrida": [len(t) for t in judged],
        "campos": FIELDS,
        "diferencias": sent,
        "diferencias_no_enviadas": len(diffs) - len(sent),
    }
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, credentials=creds)
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=MODEL,
        contents=json.dumps(payload, ensure_ascii=False),
        config=types.GenerateContentConfig(
            system_instruction=INSTRUCCIONES,
            response_mime_type="application/json",
            response_schema=Veredicto,
        ),
    )
    judge_s = time.perf_counter() - t0
    out = resp.parsed or Veredicto.model_validate_json(resp.text)

    u = resp.usage_metadata
    tok_in = u.prompt_token_count or 0
    tok_out = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    cost = (tok_in * PRICE_IN + tok_out * PRICE_OUT) / 1e6

    print(f"\nVeredicto del juez: {out.veredicto}")
    print(out.resumen)
    print(f"Juez: {judge_s:.1f}s · {tok_in:,} tokens de entrada, {tok_out:,} de salida "
          f"(incluye razonamiento) · USD {cost:.5f}")

    by_id = {d["id"]: d for d in sent}
    if out.diferencias:
        display(pd.DataFrame([
            {"id": x.id, "renglon": by_id.get(x.id, {}).get("renglon"),
             "campo": by_id.get(x.id, {}).get("campo"),
             "valores_por_corrida": by_id.get(x.id, {}).get("valores_por_corrida"),
             "materialidad": x.materialidad, "motivo": x.motivo}
            for x in out.diferencias
        ]).set_index("id"))

    # 3) Controles sobre el juez: reglas que se verifican sin LLM.
    judged_ids = {x.id for x in out.diferencias}
    expected = ("IDEMPOTENTE" if not diffs else
                "NO_IDEMPOTENTE" if any(x.materialidad == "MATERIAL" for x in out.diferencias) else
                "IDEMPOTENTE_CON_DIFERENCIAS_COSMETICAS")
    problems = []
    if out.veredicto != expected:
        problems.append(f"el veredicto no coincide con sus propias clasificaciones (debería ser {expected})")
    if judged_ids != set(by_id):
        invented = judged_ids - set(by_id)
        problems.append(f"clasificó {len(judged_ids & set(by_id))} de {len(by_id)} diferencias"
                        + (f" e inventó {len(invented)}" if invented else ""))
    for x in out.diferencias:
        d = by_id.get(x.id)
        if d and x.materialidad == "COSMETICA" and (d["campo"] in NUMERIC or d["campo"] == "(renglón)"):
            problems.append(f"marcó cosmética la diferencia {x.id} ({d['campo']}): un número distinto "
                            "o un renglón faltante siempre es material")
    if len(diffs) > len(sent):
        problems.append(f"{len(diffs) - len(sent)} diferencias no se enviaron al juez (MAX_DIFFS)")
    print("\nControles sobre el juez:", "OK" if not problems else "")
    for p in problems:
        print(" -", p)

In [ ]:
import re, unicodedata

def key(s):
    """Versión comparable: sin acentos, minúsculas, solo letras y números."""
    s = unicodedata.normalize("NFKD", s.translate(GREEK))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return re.sub(r"[^a-z0-9]", "", s)

In [ ]:
# --- Extracción real: 3 PDFs -> 3 DataFrames -> 3 CSV ------------------------------------------
# Usa el flujo de las celdas anteriores: el parser de la primera celda y split_pdf,
# process_chunk, to_table y quality de la celda de medición.
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

PDFS = [
    "/content/detalle_(1)_ANONIMIZADO_sin_afiliado.pdf",
    "/content/detalle_(2)_ANONIMIZADO_sin_afiliado.pdf",
    "/content/detalle_ANONIMIZADO_sin_afiliado.pdf",
]
MAX_WORKERS = 1      # requests en paralelo por PDF; bajalo a 1 si aparecen errores 429
SHOW_HEADERS = False

COLS = (["archivo", "pagina"] + FIELDS
        + ["codigo", "descripcion", "fecha_dt", "total_ok", "conf_min", "y0", "y1"])


def extract_pdf(path):
    """OCR del PDF completo (tramos en paralelo) y tabla de renglones. Devuelve (df, resumen)."""
    t0 = time.perf_counter()
    n_pages, chunks = split_pdf(path, PAGES_PER_REQUEST)
    with ThreadPoolExecutor(MAX_WORKERS) as pool:
        results = list(pool.map(lambda c: process_chunk(c[1]), chunks))
    ocr_s = time.perf_counter() - t0

    docs = []
    for (offset, _), (doc, *_rest) in zip(chunks, results):
        for p in doc.get("pages", []):
            p["pageNumber"] = offset + p.get("pageNumber", 0)  # número de página en el PDF completo
        docs.append((0, doc))

    df = to_table(docs)
    if not df.empty:
        df.insert(0, "archivo", Path(path).name)
        df = df.sort_values(["pagina", "y0"]).reset_index(drop=True)
        df = df[[c for c in COLS if c in df.columns]]

    summary = {
        "archivo": Path(path).name, "paginas": n_pages, "requests": len(chunks),
        "reintentos": sum(r[3] for r in results),
        "ocr_s": round(ocr_s, 1), "total_s": round(time.perf_counter() - t0, 1),
        "costo_usd": round(n_pages * PRICE_TIER1 / 1000, 4),
        **quality(docs, df),
    }
    return df, summary


dfs, summaries = {}, []
for path in PDFS:
    name = Path(path).name
    try:
        df, summary = extract_pdf(path)
    except Exception as exc:  # un PDF que falla no frena a los demás
        print(f"{name}: falló ({type(exc).__name__}: {exc})")
        summaries.append({"archivo": name, "error": type(exc).__name__})
        continue
    out = Path(path).with_suffix(".csv")
    df.to_csv(out, index=False, date_format="%Y-%m-%d")
    dfs[Path(path).stem] = df
    summaries.append({**summary, "csv": str(out)})
    print(f"{name}: {summary['renglones']} renglones → {out}")

display(pd.DataFrame(summaries).set_index("archivo"))
ok = [s for s in summaries if "error" not in s]
print(f"\nTotal: {sum(s['paginas'] for s in ok)} páginas · {sum(s['renglones'] for s in ok)} renglones · "
      f"USD {sum(s['costo_usd'] for s in ok):.4f} de OCR")

In [ ]:
# --- Exploración de los 3 CSV extraídos ------------------------------------------------------
from decimal import Decimal, InvalidOperation

import matplotlib.pyplot as plt
import pandas as pd

CSVS = [
    "/content/detalle_(1)_ANONIMIZADO_sin_afiliado.csv",
    "/content/detalle_(2)_ANONIMIZADO_sin_afiliado.csv",
    "/content/detalle_ANONIMIZADO_sin_afiliado.csv",
]
TOP = 200
pd.set_option("display.max_colwidth", 60)

# Todo como texto: agrupar por el valor exacto del CSV ("144283.00") evita sorpresas de tipos.
raw = pd.concat([pd.read_csv(p, dtype=str, keep_default_na=False) for p in CSVS], ignore_index=True)


def dec(s):
    """'144283.00' -> Decimal; vacío o ilegible -> None."""
    try:
        return Decimal(s) if s else None
    except InvalidOperation:
        return None


def is_num(v):
    return isinstance(v, Decimal)


data = raw.copy()
for c in ["der", "hon", "total"]:
    data[c] = data[c].map(dec)
data["fecha_dt"] = pd.to_datetime(data["fecha_dt"], errors="coerce")

# --- Panorama -------------------------------------------------------------------------------
print("== Panorama ==")
overview = data.groupby("archivo").agg(
    renglones=("archivo", "size"), paginas=("pagina", "nunique"),
    desde=("fecha_dt", "min"), hasta=("fecha_dt", "max"),
)
overview["total_facturado"] = data.groupby("archivo")["total"].agg(lambda s: sum(v for v in s if is_num(v)))
display(overview)
campos = ["nro_int", "nro_afiliado", "nro_bono", "profesional", "servicio", "fecha", "paciente",
          "prestacion", "cant", "der", "hon", "total"]
print("% vacío por campo:", {c: round(100 * (raw[c].str.strip() == "").mean(), 1) for c in campos if c in raw})

# --- 1. Sumas incongruentes -----------------------------------------------------------------
print("\n== 1. Sumas incongruentes: Der. + Hon. ≠ Total ==")
data["diferencia"] = [t - (d + h) if all(map(is_num, (d, h, t))) else None
                      for d, h, t in zip(data["der"], data["hon"], data["total"])]
verifiable = data["diferencia"].map(is_num)
bad = data[data["diferencia"].map(lambda v: is_num(v) and v != 0)]
print(f"{len(bad)} de {int(verifiable.sum())} renglones no cierran "
      f"(suma de diferencias: {sum(abs(v) for v in bad['diferencia']):,.2f}) · "
      f"{int((~verifiable).sum())} sin los 3 importes, no se pueden verificar")
if len(bad):
    display(bad.assign(abs_dif=bad["diferencia"].map(abs)).sort_values("abs_dif", ascending=False)
            [["archivo", "pagina", "nro_int", "fecha", "prestacion", "cant", "der", "hon", "total", "diferencia"]])

# --- 2. Duplicados en el mismo día ----------------------------------------------------------
print("\n== 2. Duplicados en el mismo día ==")
print("Paciente y afiliado vienen tachados: son candidatos con los campos visibles, no duplicados confirmados.\n")


def dup_groups(df, keys):
    """Grupos con más de un renglón para la clave dada."""
    g = df.groupby(keys).agg(
        veces=("archivo", "size"),
        descripcion=("descripcion", "first"),
        nros_int=("nro_int", lambda s: ", ".join(s)),
        archivos=("archivo", "nunique"),
    )
    return g[g["veces"] > 1].sort_values("veces", ascending=False)


# a) Cada renglón trae su propio Nro. Int.: que se repita es la señal más fuerte de un renglón
#    facturado dos veces (o del mismo detalle cargado en dos archivos).
with_nro = raw[raw["nro_int"] != ""]
rep = with_nro[with_nro.duplicated("nro_int", keep=False)].sort_values(["nro_int", "archivo"])
visible = ["fecha", "codigo", "servicio", "cant", "der", "hon", "total"]
same = rep.groupby("nro_int")[visible].nunique().max(axis=1).eq(1)
print(f"a) Mismo Nro. Int. más de una vez: {rep['nro_int'].nunique()} números en {len(rep)} renglones "
      f"({int(same.sum())} con todos los datos iguales · "
      f"{int(rep.groupby('nro_int')['archivo'].nunique().gt(1).sum())} repartidos entre archivos distintos)")
if len(rep):
    display(rep[["archivo", "pagina", "nro_int"] + visible + ["descripcion"]].head(TOP * 2))

# b) y c) Misma fecha y misma práctica, con distinto Nro. Int.
base = raw[(raw["fecha"] != "") & (raw["codigo"] != "")]
b = dup_groups(base, ["fecha", "codigo", "cant", "der", "hon", "total"])
c = dup_groups(base, ["fecha", "codigo"])
print(f"\nb) Misma fecha, código, cantidad e importes: {len(b)} grupos · "
      f"{int((b['veces'] - 1).sum())} renglones de más")
display(b.head(TOP))
print(f"\nc) Misma fecha y código, cualquier importe: {len(c)} grupos · "
      f"{int((c['veces'] - 1).sum())} renglones de más")
display(c.head(TOP))
print("\nCódigos que más se repiten en el mismo día (criterio b):")
display(b.groupby(level="codigo")["veces"]
         .agg(grupos="size", renglones_de_mas=lambda s: int((s - 1).sum()))
         .sort_values("renglones_de_mas", ascending=False).head(TOP))

# --- 3. Categóricas -------------------------------------------------------------------------
print("\n== 3. Categóricas ==")
prest = (raw[raw["codigo"] != ""].groupby("codigo")
         .agg(renglones=("codigo", "size"),
              descripcion=("descripcion", lambda s: s.value_counts().index[0]),
              descripciones_distintas=("descripcion", "nunique"))
         .sort_values("renglones", ascending=False))
print(f"Prestaciones: {len(prest)} códigos distintos · "
      f"{int((prest['descripciones_distintas'] > 1).sum())} códigos con más de una descripción")
display(prest.head(TOP))

prest.to_csv("prestaciones.csv")
serv = raw["servicio"].replace("", "(vacío)").value_counts()
prof = raw["profesional"].replace("", "(vacío)").value_counts()
print(f"\nServicio: {len(serv)} valores")
display(serv.to_frame("renglones").head(TOP))
print(f"\nProfesional efector: {len(prof)} valores (la columna viene tachada; solo cuenta lo que quedó legible)")
display(prof.to_frame("renglones").head(TOP))

labels = (prest.index.to_series() + " " + prest["descripcion"]).str.slice(0, 40)
charts = [
    ("Prestaciones (por código)", pd.Series(prest["renglones"].values, index=labels)),
    ("Servicio", serv),
    ("Profesional efector", prof),
]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (title, counts) in zip(axes, charts):
    counts.head(TOP).iloc[::-1].plot.barh(ax=ax)
    ax.set_title(title)
    ax.set_xlabel("renglones")
plt.tight_layout()
plt.show()

In [ ]:
# --- Estudios de bajo volumen y posibles duplicados (versión simple) ---------------------------
import pandas as pd

CSVS = [
    "/content/detalle_(1)_ANONIMIZADO_sin_afiliado.csv",
    "/content/detalle_(2)_ANONIMIZADO_sin_afiliado.csv",
    "/content/detalle_ANONIMIZADO_sin_afiliado.csv",
]
# Códigos que son el mismo estudio. El resto: cada código es su propia familia.
FAMILIAS = {
    "180161": "ECOGRAFIA TRANSVAGINAL", "861219": "ECOGRAFIA TRANSVAGINAL",
    "180106": "ECOGRAFIA MAMARIA",      "861220": "ECOGRAFIA MAMARIA",
    "341018": "UROTOMOGRAFIA",          "341177": "UROTOMOGRAFIA",
    "341012": "TAC OTROS ORGANOS",      "3410121": "TAC OTROS ORGANOS",
    "3410123": "TAC OTROS ORGANOS",     "3410126": "TAC OTROS ORGANOS",
}
UMBRAL = 1.5  # bajo volumen: los días que aparece, aparece en promedio menos de 1,5 veces

df = pd.concat([pd.read_csv(p, dtype=str, keep_default_na=False) for p in CSVS], ignore_index=True)
df = df[(df["fecha"] != "") & (df["codigo"] != "")].copy()
df["familia"] = df["codigo"].map(FAMILIAS).fillna(df["codigo"])

# 1) Volumen de cada estudio: renglones / días en que aparece.
vol = df.groupby("familia").agg(
    descripcion=("descripcion", lambda s: s.value_counts().index[0]),
    renglones=("codigo", "size"),
    dias=("fecha", "nunique"),
)
vol["por_dia"] = (vol["renglones"] / vol["dias"]).round(2)
vol["bajo_volumen"] = vol["por_dia"] < UMBRAL
display(vol.sort_values("por_dia"))

# 2) Posibles duplicados: estudios de bajo volumen que aparecen 2+ veces el mismo día.
bajos = vol.index[vol["bajo_volumen"]]
cand = (df[df["familia"].isin(bajos)]
        .groupby(["fecha", "familia"])
        .agg(descripcion=("descripcion", "first"),
             veces=("codigo", "size"),
             mismo_importe=("total", lambda s: s.duplicated().any()),
             nros_int=("nro_int", lambda s: ", ".join(s))))
cand = cand[cand["veces"] >= 2].sort_values(["mismo_importe", "veces"], ascending=False)
print(f"{len(cand)} posibles duplicados en estudios de bajo volumen")
display(cand)